# Train GNN Binding Affinity Model on PDBbind v2020

This notebook trains a **Graph Isomorphism Network (GIN)** to predict protein-ligand
binding affinity (pKd) from 3D molecular complexes. The model is trained on the
**PDBbind v2020 refined set** and uses the same featurization pipeline as the
backend inference service (`backend/services/gnn_affinity.py`).

**Architecture:**
- 3-layer GIN with batch normalization
- Concat(mean-pool, max-pool) graph readout
- MLP regression head predicting scalar pKd

**Designed to run on Google Colab with a GPU runtime.**

In [ ]:
# ============================================================
# Install dependencies
# ============================================================
import subprocess, sys

def pip_install(*packages):
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Install PyTorch Geometric and its dependencies
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

pip_install(
    "torch-geometric",
    "rdkit",
    "biopython",
    "scipy",
)

# Verify imports
import torch_geometric
from rdkit import Chem
from Bio.PDB import PDBParser
import scipy

print(f"torch-geometric version: {torch_geometric.__version__}")
print(f"RDKit version: {Chem.rdBase.rdkitVersion}")
print(f"SciPy version: {scipy.__version__}")
print("All dependencies installed successfully.")

## 1. Dataset: PDBbind v2020 Refined Set

You need **two downloads** from http://www.pdbbind.org.cn (free registration):

| # | File | Size | What it contains |
|---|------|------|------------------|
| **1** | Index files of PDBbind | 3.1 MB | pKd/pKi labels for every complex |
| **3** | Protein-ligand complexes: The refined set | 658 MB | 5316 PDB + SDF files |

Upload both to Google Drive, then extract them.

In [ ]:
# ============================================================
# Locate PDBbind v2020 refined set
# ============================================================
import os
from pathlib import Path

# ── Step 1: Mount Google Drive ──
# Uncomment these two lines once your files are uploaded to Drive:
# from google.colab import drive
# drive.mount('/content/drive')

# ── Step 2: Set paths to your extracted archives ──
# After downloading from pdbbind.org.cn, you should have two archives:
#   - Index files      → extract to get a folder with INDEX_*.2020 files
#   - Refined set      → extract to get folders like 1a1e/, 1a28/, etc.
#
# Point these to wherever you extracted them in Drive:
PDBBIND_DIR = Path("/content/drive/MyDrive/pdbbind/refined-set")
INDEX_DIR   = Path("/content/drive/MyDrive/pdbbind/index")

# The index file with pKd values for the refined set:
INDEX_FILE = INDEX_DIR / "INDEX_refined_data.2020"

# ── Fallback: auto-detect common structures ──
# PDBbind sometimes nests: refined-set/refined-set/index/... or refined-set/index/...
if not INDEX_FILE.exists():
    # Try: index is inside the refined-set folder
    alt = PDBBIND_DIR / "index" / "INDEX_refined_data.2020"
    if alt.exists():
        INDEX_FILE = alt
        INDEX_DIR = alt.parent

# ── Check if data exists, otherwise use mock ──
USE_MOCK = not INDEX_FILE.exists()

if USE_MOCK:
    print("=" * 60)
    print("PDBbind data not found. Using mock dataset for testing.")
    print("=" * 60)
    print()
    print("For real training, do this:")
    print("  1. Register at http://www.pdbbind.org.cn")
    print("  2. Download row #1 (Index files, 3.1 MB)")
    print("  3. Download row #3 (Refined set, 658 MB)")
    print("  4. Upload both zips to Google Drive")
    print("  5. Extract them and update PDBBIND_DIR / INDEX_DIR above")
    print()

    # Create mock dataset for pipeline testing
    MOCK_DIR = Path("/content/mock_pdbbind")
    (MOCK_DIR / "index").mkdir(parents=True, exist_ok=True)

    import random
    random.seed(42)

    # Mock index (real PDBbind format: "code  res  year  pkd  Kd/Ki  //  ref")
    mock_entries = []
    for i in range(100):
        pdb_code = f"mock{i:04d}"
        pkd = round(random.uniform(2.0, 12.0), 2)
        mock_entries.append(f"{pdb_code}  2.00  2020  {pkd}  Kd={10**(-pkd)*1e6:.1f}uM  //  mock entry {i}")

    mock_index_path = MOCK_DIR / "index" / "INDEX_refined_data.2020"
    mock_index_path.write_text("\n".join(mock_entries) + "\n")

    # Create mock PDB and SDF files
    from rdkit import Chem
    from rdkit.Chem import AllChem

    smiles_pool = [
        "CC(=O)Oc1ccccc1C(=O)O",       # aspirin
        "CC(C)Cc1ccc(cc1)C(C)C(=O)O",   # ibuprofen
        "CC(=O)Nc1ccc(O)cc1",            # acetaminophen
        "c1ccc(cc1)C(=O)O",              # benzoic acid
        "OC(=O)c1cccnc1",               # nicotinic acid
    ]
    aa_names = ["ALA", "GLY", "LEU", "VAL", "SER", "THR", "ASP", "GLU", "LYS", "ARG"]

    for entry_line in mock_entries:
        pdb_code = entry_line.split()[0]
        entry_dir = MOCK_DIR / pdb_code
        entry_dir.mkdir(exist_ok=True)

        # Random ligand with 3D coords
        smi = random.choice(smiles_pool)
        mol = Chem.MolFromSmiles(smi)
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol, randomSeed=int(pdb_code[-4:]))
        AllChem.MMFFOptimizeMolecule(mol)
        (entry_dir / f"{pdb_code}_ligand.sdf").write_text(Chem.MolToMolBlock(mol))

        # Random protein pocket around ligand center
        conf = mol.GetConformer()
        cx = sum(conf.GetAtomPosition(a).x for a in range(mol.GetNumAtoms())) / mol.GetNumAtoms()
        cy = sum(conf.GetAtomPosition(a).y for a in range(mol.GetNumAtoms())) / mol.GetNumAtoms()
        cz = sum(conf.GetAtomPosition(a).z for a in range(mol.GetNumAtoms())) / mol.GetNumAtoms()

        pdb_lines = []
        for j in range(30):
            x, y, z = cx + random.uniform(-8, 8), cy + random.uniform(-8, 8), cz + random.uniform(-8, 8)
            aa = random.choice(aa_names)
            elem = random.choice(["C", "N", "O", "C", "C"])
            atom_name = {"C": "CA", "N": "N", "O": "O"}[elem]
            pdb_lines.append(
                f"ATOM  {j+1:5d} {atom_name:<4s} {aa:3s} A{j+1:4d}    "
                f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           {elem:>2s}"
            )
        pdb_lines.append("END")
        (entry_dir / f"{pdb_code}_protein.pdb").write_text("\n".join(pdb_lines) + "\n")

    PDBBIND_DIR = MOCK_DIR
    INDEX_FILE = mock_index_path
    INDEX_DIR = mock_index_path.parent
    print(f"Created {len(mock_entries)} mock complexes in {MOCK_DIR}")

else:
    # Count actual complex directories
    complex_dirs = [d for d in PDBBIND_DIR.iterdir() if d.is_dir() and d.name != "index"]
    print(f"PDBbind refined set found: {PDBBIND_DIR}")
    print(f"Index file: {INDEX_FILE}")
    print(f"Complex directories: {len(complex_dirs)}")

In [ ]:
# ============================================================
# Parse PDBbind index file to get PDB code -> pKd mapping
# ============================================================

def parse_pdbbind_index(index_path: Path) -> dict:
    """Parse PDBbind INDEX_refined_data.2020 and return {pdb_code: pkd_value}.

    Real PDBbind format (whitespace-separated):
        1a28  2.80  1997  4.41  Ki=39uM  //  (reference info)
        ^^^^                ^^^^
        col 0 = PDB code    col 3 = -log(Kd/Ki) = pKd

    Lines starting with '#' are comments and are skipped.
    """
    entries = {}
    with open(index_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            # Split on '//' first to separate data from reference
            main_part = line.split("//")[0].strip()
            tokens = main_part.split()
            if len(tokens) < 4:
                continue
            pdb_code = tokens[0]
            try:
                pkd = float(tokens[3])
            except ValueError:
                continue
            entries[pdb_code] = pkd
    return entries

pkd_map = parse_pdbbind_index(INDEX_FILE)
print(f"Parsed {len(pkd_map)} entries from index file.")

# Show distribution
if pkd_map:
    values = list(pkd_map.values())
    print(f"pKd range: [{min(values):.2f}, {max(values):.2f}]")
    print(f"pKd mean:  {sum(values)/len(values):.2f}")
    print()
    # Show a few examples
    for code, pkd in list(pkd_map.items())[:5]:
        print(f"  {code}: pKd = {pkd}")

## 2. Featurization

The featurization code below is **identical** to the backend service
(`backend/services/gnn_affinity.py`). Each protein-ligand complex is
converted into a heterogeneous graph with:

- **47-dim node features** (element one-hot, hybridization, charge, aromaticity, ring, neighbors, protein flag, residue type)
- **24-dim edge features** (21 RBF distance bins + 3-dim edge type one-hot: intra-ligand, intra-protein, inter-molecular)

In [ ]:
# ============================================================
# Featurization (EXACT copy from backend/services/gnn_affinity.py)
# ============================================================
import numpy as np
import torch
import torch_geometric.data as pyg_data
from torch_geometric.data import Data
from rdkit import Chem
from Bio.PDB import PDBParser
from scipy.spatial.distance import cdist
import io
import warnings

# ─── Constants ───────────────────────────────────────────────────────────────

# One-hot encoding for common elements (top 10 in PDBbind)
ELEMENT_LIST = [6, 7, 8, 16, 15, 9, 17, 35, 53, 5]  # C,N,O,S,P,F,Cl,Br,I,B

# 20 standard amino acids
AA_LIST = [
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY",
    "HIS", "ILE", "LEU", "LYS", "MET", "PHE", "PRO", "SER",
    "THR", "TRP", "TYR", "VAL",
]
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_LIST)}

# Hybridization mapping (RDKit)
HYBRIDIZATION_LIST = ["SP", "SP2", "SP3", "SP3D", "SP3D2"]

# RBF parameters for edge distance encoding
RBF_CENTERS = np.linspace(0.0, 10.0, 21)
RBF_GAMMA = 5.0

# Node feature dimensionality: 10 (element) + 5 (hybrid) + 1 (charge) + 1 (arom) + 1 (ring) + 1 (heavy_neighbors) + 1 (is_protein) + 20 (AA) + 7 padding = 47
NODE_DIM = 47
EDGE_DIM = 24  # 21 (RBF distance) + 3 (edge type one-hot: intra-lig, intra-prot, inter)


# ─── Helper functions ────────────────────────────────────────────────────────

def _rbf_encode(distance: float) -> np.ndarray:
    """Radial Basis Function encoding of a single distance."""
    return np.exp(-RBF_GAMMA * (distance - RBF_CENTERS) ** 2).astype(np.float32)


def _atom_features(
    atomic_num: int,
    hybridization: str | None = None,
    formal_charge: int = 0,
    is_aromatic: bool = False,
    is_in_ring: bool = False,
    num_heavy_neighbors: int = 0,
    is_protein: bool = False,
    residue_name: str | None = None,
) -> np.ndarray:
    """Compute node feature vector (47 dims)."""
    feat = np.zeros(NODE_DIM, dtype=np.float32)
    idx = 0

    # Element one-hot (10)
    if atomic_num in ELEMENT_LIST:
        feat[ELEMENT_LIST.index(atomic_num)] = 1.0
    idx += 10

    # Hybridization one-hot (5)
    if hybridization and hybridization in HYBRIDIZATION_LIST:
        feat[idx + HYBRIDIZATION_LIST.index(hybridization)] = 1.0
    idx += 5

    # Formal charge (1)
    feat[idx] = float(formal_charge)
    idx += 1

    # Aromaticity (1)
    feat[idx] = 1.0 if is_aromatic else 0.0
    idx += 1

    # Ring membership (1)
    feat[idx] = 1.0 if is_in_ring else 0.0
    idx += 1

    # Num heavy neighbors (1)
    feat[idx] = float(num_heavy_neighbors)
    idx += 1

    # Is protein flag (1)
    feat[idx] = 1.0 if is_protein else 0.0
    idx += 1

    # Residue type one-hot (20)
    if residue_name and residue_name in AA_TO_IDX:
        feat[idx + AA_TO_IDX[residue_name]] = 1.0
    idx += 20

    return feat


def featurize_complex(pdb_text: str, ligand_sdf: str) -> Data | None:
    """Convert protein PDB + ligand SDF strings into a PyG Data graph.

    This function accepts raw text (not file paths) so it can be reused
    identically in the training notebook and inference service.

    Returns a torch_geometric.data.Data object or None on failure.
    """
    try:
        # --- Parse ligand ---
        mol = Chem.MolFromMolBlock(ligand_sdf, removeHs=False)
        if mol is None:
            mol = Chem.MolFromMolBlock(ligand_sdf, removeHs=True)
        if mol is None:
            return None

        # Get ligand conformer coordinates
        conf = mol.GetConformer()
        lig_coords = []
        lig_features = []
        for atom in mol.GetAtoms():
            pos = conf.GetAtomPosition(atom.GetIdx())
            lig_coords.append([pos.x, pos.y, pos.z])
            hyb = str(atom.GetHybridization()) if atom.GetHybridization() else None
            lig_features.append(_atom_features(
                atomic_num=atom.GetAtomicNum(),
                hybridization=hyb,
                formal_charge=atom.GetFormalCharge(),
                is_aromatic=atom.GetIsAromatic(),
                is_in_ring=atom.IsInRing(),
                num_heavy_neighbors=len([n for n in atom.GetNeighbors() if n.GetAtomicNum() > 1]),
                is_protein=False,
            ))
        lig_coords = np.array(lig_coords)
        n_lig = len(lig_features)

        # --- Parse protein binding pocket ---
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            parser = PDBParser(QUIET=True)
            structure = parser.get_structure("protein", io.StringIO(pdb_text))

        prot_atoms = []
        for atom in structure.get_atoms():
            if atom.element.strip() in ("H", ""):
                continue
            prot_atoms.append(atom)

        if not prot_atoms:
            return None

        prot_coords_all = np.array([a.get_vector().get_array() for a in prot_atoms])

        # Extract binding pocket: protein atoms within 10A of any ligand atom
        dist_matrix = cdist(prot_coords_all, lig_coords)
        pocket_mask = dist_matrix.min(axis=1) < 10.0
        pocket_indices = np.where(pocket_mask)[0]

        if len(pocket_indices) == 0:
            return None

        pocket_atoms = [prot_atoms[i] for i in pocket_indices]
        pocket_coords = prot_coords_all[pocket_indices]

        # Protein node features
        prot_features = []
        for atom in pocket_atoms:
            elem = atom.element.strip()
            atomic_num = Chem.GetPeriodicTable().GetAtomicNumber(elem) if elem else 6
            residue = atom.get_parent()
            res_name = residue.get_resname() if residue else None
            prot_features.append(_atom_features(
                atomic_num=atomic_num,
                is_protein=True,
                residue_name=res_name,
            ))
        n_prot = len(prot_features)

        # --- Build graph ---
        all_features = np.array(lig_features + prot_features, dtype=np.float32)
        all_coords = np.vstack([lig_coords, pocket_coords])

        edge_src = []
        edge_dst = []
        edge_features = []

        # Edge type one-hot: [intra_ligand, intra_protein, inter]

        # Intra-ligand bonds (from RDKit)
        for bond in mol.GetBonds():
            i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            dist = np.linalg.norm(all_coords[i] - all_coords[j])
            rbf = _rbf_encode(dist)
            etype = np.array([1, 0, 0], dtype=np.float32)
            ef = np.concatenate([rbf, etype])
            edge_src.extend([i, j])
            edge_dst.extend([j, i])
            edge_features.extend([ef, ef])

        # Intra-protein bonds (< 1.6A between pocket atoms)
        if n_prot > 1:
            prot_dist = cdist(pocket_coords, pocket_coords)
            pi, pj = np.where((prot_dist < 1.6) & (prot_dist > 0.1))
            for ii, jj in zip(pi, pj):
                dist = prot_dist[ii, jj]
                rbf = _rbf_encode(dist)
                etype = np.array([0, 1, 0], dtype=np.float32)
                ef = np.concatenate([rbf, etype])
                edge_src.append(n_lig + ii)
                edge_dst.append(n_lig + jj)
                edge_features.append(ef)

        # Protein-ligand interactions (< 5A)
        cross_dist = cdist(lig_coords, pocket_coords)
        li, pi = np.where(cross_dist < 5.0)
        for l_idx, p_idx in zip(li, pi):
            dist = cross_dist[l_idx, p_idx]
            rbf = _rbf_encode(dist)
            etype = np.array([0, 0, 1], dtype=np.float32)
            ef = np.concatenate([rbf, etype])
            # Bidirectional
            edge_src.extend([l_idx, n_lig + p_idx])
            edge_dst.extend([n_lig + p_idx, l_idx])
            edge_features.extend([ef, ef])

        if not edge_src:
            return None

        x = torch.tensor(all_features, dtype=torch.float32)
        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        edge_attr = torch.tensor(np.array(edge_features, dtype=np.float32), dtype=torch.float32)
        pos = torch.tensor(all_coords, dtype=torch.float32)

        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, pos=pos)

    except Exception as e:
        print(f"Warning: featurize_complex failed: {e}")
        return None


print(f"NODE_DIM = {NODE_DIM}, EDGE_DIM = {EDGE_DIM}")
print("Featurization functions loaded.")

In [ ]:
# ============================================================
# Featurize all complexes and cache to disk
# ============================================================
from pathlib import Path
from tqdm.auto import tqdm

CACHE_DIR = Path("/content/featurized_graphs")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

dataset = []  # list of (Data, pkd)
skipped = 0

pdb_codes = sorted(pkd_map.keys())
print(f"Processing {len(pdb_codes)} complexes...")

for pdb_code in tqdm(pdb_codes, desc="Featurizing"):
    cache_path = CACHE_DIR / f"{pdb_code}.pt"

    # Check cache first
    if cache_path.exists():
        cached = torch.load(cache_path, weights_only=False)
        dataset.append((cached["data"], cached["pkd"]))
        continue

    # Read PDB and SDF files
    entry_dir = PDBBIND_DIR / pdb_code
    pdb_path = entry_dir / f"{pdb_code}_protein.pdb"

    # PDBbind includes ligands in both .sdf and .mol2 — prefer .sdf
    sdf_path = entry_dir / f"{pdb_code}_ligand.sdf"
    if not sdf_path.exists():
        # Some entries only have .mol2; try converting via RDKit
        mol2_path = entry_dir / f"{pdb_code}_ligand.mol2"
        if mol2_path.exists():
            try:
                mol = Chem.MolFromMol2File(str(mol2_path), removeHs=False)
                if mol is not None:
                    sdf_text = Chem.MolToMolBlock(mol)
                    sdf_path.write_text(sdf_text)  # cache the conversion
            except Exception:
                pass

    if not pdb_path.exists() or not sdf_path.exists():
        skipped += 1
        continue

    pdb_text = pdb_path.read_text()
    sdf_text = sdf_path.read_text()

    # Featurize
    data = featurize_complex(pdb_text, sdf_text)
    if data is None:
        skipped += 1
        continue

    pkd = pkd_map[pdb_code]
    data.y = torch.tensor([pkd], dtype=torch.float32)

    # Cache to disk
    torch.save({"data": data, "pkd": pkd}, cache_path)
    dataset.append((data, pkd))

print(f"\nSuccessfully featurized: {len(dataset)} complexes")
print(f"Skipped: {skipped} complexes")
print(f"Cache directory: {CACHE_DIR}")

# Quick stats on the dataset
if dataset:
    pkd_values = [pkd for _, pkd in dataset]
    print(f"\npKd range: [{min(pkd_values):.2f}, {max(pkd_values):.2f}]")
    print(f"pKd mean: {np.mean(pkd_values):.2f}, std: {np.std(pkd_values):.2f}")

## 3. Model Definition

A 3-layer **Graph Isomorphism Network (GIN)** with batch normalization.
Graph-level readout is `concat(mean_pool, max_pool)` followed by an MLP head
that predicts a scalar pKd value.

In [ ]:
# ============================================================
# Model: BindingAffinityGNN
# ============================================================
import torch
import torch.nn as nn
from torch_geometric.nn import GINConv, global_mean_pool, global_max_pool


class BindingAffinityGNN(nn.Module):
    """3-layer GIN with batch norm for pKd prediction."""

    def __init__(self, node_dim=47, edge_dim=24, hidden_dim=128):
        super().__init__()
        self.node_encoder = nn.Linear(node_dim, hidden_dim)

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for _ in range(3):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        # MLP head: concat(mean_pool, max_pool) -> scalar
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, data):
        x = self.node_encoder(data.x)
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, data.edge_index)
            x = bn(x)
            x = x.relu()

        batch = data.batch if hasattr(data, 'batch') and data.batch is not None else torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x_cat = torch.cat([x_mean, x_max], dim=-1)

        return self.head(x_cat).squeeze(-1)


# Instantiate and print model summary
model = BindingAffinityGNN(node_dim=NODE_DIM, edge_dim=EDGE_DIM, hidden_dim=128)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: BindingAffinityGNN")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(model)

## 4. Training

- **Split:** 80/10/10 (train / validation / test)
- **Optimizer:** Adam with lr=1e-3
- **Scheduler:** Cosine annealing
- **Loss:** MSE
- **Early stopping:** patience=20 epochs on validation RMSE

In [ ]:
# ============================================================
# Training loop
# ============================================================
import torch
import torch.nn as nn
import numpy as np
from torch_geometric.loader import DataLoader
import math
import copy

# ─── Prepare data splits ────────────────────────────────────────────────────

# Attach pKd labels to Data objects
data_list = []
for data_obj, pkd in dataset:
    data_obj.y = torch.tensor([pkd], dtype=torch.float32)
    data_list.append(data_obj)

# Shuffle with fixed seed for reproducibility
rng = np.random.RandomState(42)
indices = rng.permutation(len(data_list))

n_total = len(data_list)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)
n_test = n_total - n_train - n_val

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

train_data = [data_list[i] for i in train_idx]
val_data = [data_list[i] for i in val_idx]
test_data = [data_list[i] for i in test_idx]

print(f"Dataset split: train={len(train_data)}, val={len(val_data)}, test={len(test_data)}")

# DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

# ─── Training setup ─────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

model = BindingAffinityGNN(node_dim=NODE_DIM, edge_dim=EDGE_DIM, hidden_dim=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

NUM_EPOCHS = 200
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

PATIENCE = 20
best_val_rmse = float("inf")
best_model_state = None
patience_counter = 0

train_rmse_history = []
val_rmse_history = []

# ─── Training loop ──────────────────────────────────────────────────────────

def evaluate(loader):
    """Evaluate model on a DataLoader, return RMSE."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch)
            preds.append(pred.cpu())
            targets.append(batch.y.cpu().squeeze())
    preds = torch.cat(preds)
    targets = torch.cat(targets)
    rmse = torch.sqrt(torch.mean((preds - targets) ** 2)).item()
    return rmse


print(f"\nStarting training for {NUM_EPOCHS} epochs (early stopping patience={PATIENCE})...\n")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_losses = []

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)
        loss = criterion(pred, batch.y.squeeze())
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    scheduler.step()

    # Compute metrics
    train_rmse = math.sqrt(np.mean(train_losses))
    val_rmse = evaluate(val_loader)

    train_rmse_history.append(train_rmse)
    val_rmse_history.append(val_rmse)

    # Early stopping check
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        marker = " *"
    else:
        patience_counter += 1
        marker = ""

    if epoch % 5 == 0 or epoch == 1 or patience_counter == 0:
        lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | "
              f"Train RMSE: {train_rmse:.4f} | "
              f"Val RMSE: {val_rmse:.4f} | "
              f"LR: {lr:.6f}{marker}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (patience={PATIENCE})")
        break

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nRestored best model (val RMSE = {best_val_rmse:.4f})")

# Plot training curves
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_rmse_history, label="Train RMSE")
ax.plot(val_rmse_history, label="Val RMSE")
ax.set_xlabel("Epoch")
ax.set_ylabel("RMSE (pKd)")
ax.set_title("Training and Validation RMSE")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Evaluation

Evaluate on the held-out test set: RMSE, MAE, and Pearson correlation.

In [ ]:
# ============================================================
# Evaluate on test set
# ============================================================
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = model(batch)
        all_preds.append(pred.cpu().numpy())
        all_targets.append(batch.y.cpu().squeeze().numpy())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

# Compute metrics
test_rmse = np.sqrt(np.mean((all_preds - all_targets) ** 2))
test_mae = np.mean(np.abs(all_preds - all_targets))
pearson_r, pearson_p = pearsonr(all_targets, all_preds)

print("=" * 50)
print("Test Set Evaluation")
print("=" * 50)
print(f"  RMSE:      {test_rmse:.4f} pKd")
print(f"  MAE:       {test_mae:.4f} pKd")
print(f"  Pearson R: {pearson_r:.4f} (p={pearson_p:.2e})")
print(f"  N test:    {len(all_targets)}")
print("=" * 50)

# Also compute train RMSE for export
train_rmse = evaluate(train_loader)
val_rmse = evaluate(val_loader)
print(f"\nFinal Train RMSE: {train_rmse:.4f}")
print(f"Final Val RMSE:   {val_rmse:.4f}")
print(f"Final Test RMSE:  {test_rmse:.4f}")

# ─── Scatter plot: predicted vs actual pKd ───────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(all_targets, all_preds, alpha=0.6, edgecolors="k", linewidths=0.3, s=40)

# Diagonal line (perfect prediction)
lims = [
    min(all_targets.min(), all_preds.min()) - 0.5,
    max(all_targets.max(), all_preds.max()) + 0.5,
]
ax.plot(lims, lims, "--", color="red", linewidth=1.5, label="Ideal (y=x)")

ax.set_xlabel("Actual pKd", fontsize=13)
ax.set_ylabel("Predicted pKd", fontsize=13)
ax.set_title(
    f"GNN Binding Affinity: Predicted vs Actual\n"
    f"RMSE={test_rmse:.3f}, MAE={test_mae:.3f}, R={pearson_r:.3f}",
    fontsize=13,
)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Export Model

Save the trained model weights along with configuration metadata.
The exported file can be loaded by `backend/services/gnn_affinity.py`.

In [ ]:
# ============================================================
# Export trained model
# ============================================================
import torch

export_path = "gnn_binding_affinity.pt"

torch.save({
    "state_dict": model.state_dict(),
    "hidden_dim": 128,
    "node_dim": 47,
    "edge_dim": 24,
    "train_rmse": train_rmse,
    "val_rmse": val_rmse,
    "test_rmse": test_rmse,
    "pearson_r": pearson_r,
}, export_path)

# Verify the saved file
import os
file_size = os.path.getsize(export_path) / (1024 * 1024)
print(f"Model saved to: {export_path}")
print(f"File size: {file_size:.2f} MB")

# Verify it loads correctly
checkpoint = torch.load(export_path, map_location="cpu", weights_only=False)
print(f"\nCheckpoint keys: {list(checkpoint.keys())}")
print(f"  hidden_dim: {checkpoint['hidden_dim']}")
print(f"  node_dim:   {checkpoint['node_dim']}")
print(f"  edge_dim:   {checkpoint['edge_dim']}")
print(f"  train_rmse: {checkpoint['train_rmse']:.4f}")
print(f"  val_rmse:   {checkpoint['val_rmse']:.4f}")
print(f"  test_rmse:  {checkpoint['test_rmse']:.4f}")
print(f"  pearson_r:  {checkpoint['pearson_r']:.4f}")
print("\nExport complete. Copy this file to backend/models/gnn_affinity/ for deployment.")